In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive"

DRIVE_OFFICIAL_REPO = f"{DRIVE_ROOT}/official_structured_initialization"
DRIVE_RESULTS_BACKUP = f"{DRIVE_ROOT}/official_structured_results"

LOCAL_REPO = "/content/official_structured_initialization"
LOCAL_CODE = f"{LOCAL_REPO}/pytorch-image-models-1.0.22"
LOCAL_DATA = "/content/official_structured_data"
LOCAL_RESULTS = "/content/official_structured_results"

!mkdir -p "$LOCAL_DATA" "$LOCAL_RESULTS" "$DRIVE_RESULTS_BACKUP"
!nvidia-smi


Mounted at /content/drive
Wed Apr 29 13:59:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   36C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------

In [ ]:
%cd /content

!rm -rf "$LOCAL_REPO"
!git clone --depth 1 https://github.com/osiriszjq/structured_initialization.git "$LOCAL_REPO"

%cd "$LOCAL_CODE"

!python -m pip install -U pip setuptools wheel
!python -m pip install -r requirements.txt
!python -m pip install -e .


/content
Cloning into '/content/official_structured_initialization'...
remote: Enumerating objects: 466, done.
remote: Counting objects: 100% (466/466), done.
remote: Compressing objects: 100% (371/371), done.
remote: Total 466 (delta 91), reused 461 (delta 91), pack-reused 0 (from 0)
Receiving objects: 100% (466/466), 2.87 MiB | 22.46 MiB/s, done.
Resolving deltas: 100% (91/91), done.
/content/official_structured_initialization/pytorch-image-models-1.0.22
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 79.4 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account a

In [ ]:
import subprocess
import sys
import time

def run_official_fast(
    dataset="CIFAR10",
    num_classes=10,
    init="default",
    seed=0,
    epochs=100,
    warmup_epochs=10,
    batch_size=512,
    workers=8,
):
    exp = f"{dataset.lower()}_official_{init}_seed{seed}_{epochs}ep_bs{batch_size}"

    cmd = [
        "python", "-u", "train.py", LOCAL_DATA,
        "--dataset", f"torch/{dataset}",
        "--dataset-download",
        "--input-size", "3", "224", "224",
        "--mean", "0.485", "0.456", "0.406",
        "--std", "0.229", "0.224", "0.225",
        "--seed", str(seed),
        "--num-classes", str(num_classes),
        "--model", "vit_tiny_patch16_224",
        "--model-kwargs",
        "img_size=224",
        "weight_init=skip",
        f"post_weight_init={init}",
        "--model-ema-decay", "0.9999",
        "-j", str(workers),
        "-b", str(batch_size),
        "--lr", "2e-3",
        "--layer-decay", "1.0",
        "--warmup-lr", "1e-6",
        "--min-lr", "1e-6",
        "--weight-decay", "0.05",
        "--opt", "adamw",
        "--opt-eps", "1e-8",
        "--epochs", str(epochs),
        "--sched", "cosine",
        "--warmup-epochs", str(warmup_epochs),
        "--amp",
        "--aa", "rand-m9-mstd0.5-inc1",
        "--cutmix", "1.0",
        "--mixup", "0.8",
        "--reprob", "0.25",
        "--smoothing", "0.1",
        "--drop", "0.0",
        "--color-jitter", "0.4",
        "--drop-path", "0.1",
        "--crop-pct", "0.875",
        "--pin-mem",

        # Reduce checkpoint clutter.
        "--checkpoint-hist", "1",
        "--recovery-interval", "0",

        "--output", LOCAL_RESULTS,
        "--experiment", exp,
    ]

    print("\n" + "=" * 90)
    print(f"Starting experiment: {exp}")
    print(f"Dataset={dataset}, init={init}, seed={seed}, epochs={epochs}, batch_size={batch_size}")
    print("Command:")
    print(" ".join(cmd))
    print("=" * 90 + "\n", flush=True)

    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    last_progress_time = time.time()
    recent_lines = []

    for line in process.stdout:
        line = line.rstrip()
        recent_lines.append(line)
        recent_lines = recent_lines[-8:]

        # Print important lines immediately.
        important = (
            "Train:" in line
            or "Test:" in line
            or "Epoch:" in line
            or "Acc@" in line
            or "loss" in line.lower()
            or "error" in line.lower()
            or "warning" in line.lower()
        )

        if important:
            print(line, flush=True)

        # Heartbeat every 60 seconds, useful when timm is quiet.
        if time.time() - last_progress_time > 60:
            print("\n--- still running; recent output ---")
            for l in recent_lines:
                print(l)
            print("-----------------------------------\n", flush=True)
            last_progress_time = time.time()

    return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(f"{exp} failed with exit code {return_code}")

    print(f"\nFinished experiment: {exp}\n", flush=True)

In [ ]:
%cd "$LOCAL_CODE"

for init in ["impulse"]:
    run_official_fast(
        dataset="CIFAR100",
        num_classes=100,
        init=init,
        seed=0,
        epochs=250,
        warmup_epochs=10,
        batch_size=1024,
        workers=8,
    )

/content/official_structured_initialization/pytorch-image-models-1.0.22

Starting experiment: cifar100_official_impulse_seed0_250ep_bs1024
Dataset=CIFAR100, init=impulse, seed=0, epochs=250, batch_size=1024
Command:
python -u train.py /content/official_structured_data --dataset torch/CIFAR100 --dataset-download --input-size 3 224 224 --mean 0.485 0.456 0.406 --std 0.229 0.224 0.225 --seed 0 --num-classes 100 --model vit_tiny_patch16_224 --model-kwargs img_size=224 weight_init=skip post_weight_init=impulse --model-ema-decay 0.9999 -j 8 -b 1024 --lr 2e-3 --layer-decay 1.0 --warmup-lr 1e-6 --min-lr 1e-6 --weight-decay 0.05 --opt adamw --opt-eps 1e-8 --epochs 250 --sched cosine --warmup-epochs 10 --amp --aa rand-m9-mstd0.5-inc1 --cutmix 1.0 --mixup 0.8 --reprob 0.25 --smoothing 0.1 --drop 0.0 --color-jitter 0.4 --drop-path 0.1 --crop-pct 0.875 --pin-mem --checkpoint-hist 1 --recovery-interval 0 --output /content/official_structured_results --experiment cifar100_official_impulse_seed0_250ep

In [ ]:
!mkdir -p "$DRIVE_RESULTS_BACKUP"
!cp -r "$LOCAL_RESULTS"/* "$DRIVE_RESULTS_BACKUP"/
print("Backed up results to:", DRIVE_RESULTS_BACKUP)

Backed up results to: /content/drive/MyDrive/official_structured_results


In [ ]:
# Run 10% CIFAR-10 and 10% CIFAR-100 with official structured_initialization code.
# This creates ImageFolder-style datasets so we do not need train.py to support subset-frac.

%cd "$LOCAL_CODE"

import os
import shutil
import subprocess
import time
import random
from pathlib import Path

from torchvision.datasets import CIFAR10, CIFAR100


SUBSET_FRAC = 0.10
SEEDS = [1]          # change to [1, 2] if you want two seeds
EPOCHS = 150         # use 250 if you want same epoch budget as full-data runs
BATCH_SIZE = 512     # use 1024 if GPU memory allows
WORKERS = 8


def build_cifar_10pct_imagefolder(dataset_name, seed=1, rebuild=False):
    dataset_name = dataset_name.lower()
    assert dataset_name in ["cifar10", "cifar100"]

    ds_class = CIFAR10 if dataset_name == "cifar10" else CIFAR100
    raw_root = Path(LOCAL_DATA) / "raw_torchvision"
    out_root = Path(LOCAL_DATA) / f"{dataset_name}_10pct_seed{seed}_imagefolder"

    if out_root.exists() and not rebuild:
        print(f"Using existing subset: {out_root}")
        return str(out_root)

    if out_root.exists():
        shutil.rmtree(out_root)

    train_set = ds_class(root=str(raw_root), train=True, download=True)
    test_set = ds_class(root=str(raw_root), train=False, download=True)
    classes = train_set.classes

    # Stratified 10% subset per class.
    rng = random.Random(seed)
    by_class = {i: [] for i in range(len(classes))}
    for idx, label in enumerate(train_set.targets):
        by_class[label].append(idx)

    keep_indices = []
    for label, indices in by_class.items():
        rng.shuffle(indices)
        k = max(1, int(len(indices) * SUBSET_FRAC))
        keep_indices.extend(indices[:k])

    def save_split(dataset, indices, split_name):
        for n, idx in enumerate(indices):
            img, label = dataset[idx]
            class_dir = out_root / split_name / classes[label]
            class_dir.mkdir(parents=True, exist_ok=True)
            img.save(class_dir / f"{idx:06d}.png")
            if n % 2000 == 0:
                print(f"{dataset_name} {split_name}: saved {n}/{len(indices)}")

    save_split(train_set, keep_indices, "train")
    save_split(test_set, list(range(len(test_set))), "validation")

    print(f"Built {dataset_name} 10% subset at: {out_root}")
    print(f"Train images: {len(keep_indices)} | Validation images: {len(test_set)}")
    return str(out_root)


def run_official_imagefolder_10pct(
    dataset_name,
    dataset_root,
    num_classes,
    init="default",
    seed=1,
    epochs=150,
    warmup_epochs=10,
    batch_size=512,
    workers=8,
):
    exp = f"{dataset_name}_10pct_official_{init}_seed{seed}_{epochs}ep_bs{batch_size}"

    cmd = [
        "python", "-u", "train.py", dataset_root,

        "--train-split", "train",
        "--val-split", "validation",

        "--input-size", "3", "224", "224",
        "--mean", "0.485", "0.456", "0.406",
        "--std", "0.229", "0.224", "0.225",

        "--seed", str(seed),
        "--num-classes", str(num_classes),
        "--model", "vit_tiny_patch16_224",
        "--model-kwargs",
        "img_size=224",
        "weight_init=skip",
        f"post_weight_init={init}",

        "--model-ema-decay", "0.9999",
        "-j", str(workers),
        "-b", str(batch_size),

        "--lr", "2e-3",
        "--layer-decay", "1.0",
        "--warmup-lr", "1e-6",
        "--min-lr", "1e-6",
        "--weight-decay", "0.05",
        "--opt", "adamw",
        "--opt-eps", "1e-8",
        "--epochs", str(epochs),
        "--sched", "cosine",
        "--warmup-epochs", str(warmup_epochs),
        "--amp",

        "--aa", "rand-m9-mstd0.5-inc1",
        "--cutmix", "1.0",
        "--mixup", "0.8",
        "--reprob", "0.25",
        "--smoothing", "0.1",
        "--drop", "0.0",
        "--color-jitter", "0.4",
        "--drop-path", "0.1",
        "--crop-pct", "0.875",
        "--pin-mem",

        "--checkpoint-hist", "1",
        "--recovery-interval", "0",

        "--output", LOCAL_RESULTS,
        "--experiment", exp,
    ]

    print("\n" + "=" * 90)
    print(f"Starting experiment: {exp}")
    print(" ".join(cmd))
    print("=" * 90 + "\n", flush=True)

    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    recent_lines = []
    last_progress_time = time.time()

    for line in process.stdout:
        line = line.rstrip()
        recent_lines.append(line)
        recent_lines = recent_lines[-8:]

        important = (
            "Train:" in line
            or "Test:" in line
            or "Epoch:" in line
            or "Acc@" in line
            or "loss" in line.lower()
            or "error" in line.lower()
            or "warning" in line.lower()
        )

        if important:
            print(line, flush=True)

        if time.time() - last_progress_time > 60:
            print("\n--- still running; recent output ---")
            for l in recent_lines:
                print(l)
            print("-----------------------------------\n", flush=True)
            last_progress_time = time.time()

    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"{exp} failed with exit code {return_code}")

    print(f"\nFinished experiment: {exp}\n", flush=True)


for seed in SEEDS:
    cifar10_root = build_cifar_10pct_imagefolder("cifar10", seed=seed)
    cifar100_root = build_cifar_10pct_imagefolder("cifar100", seed=seed)

    for dataset_name, dataset_root, num_classes in [
        ("cifar10", cifar10_root, 10),
        ("cifar100", cifar100_root, 100),
    ]:
        for init in ["default", "mimetic", "impulse"]:
            run_official_imagefolder_10pct(
                dataset_name=dataset_name,
                dataset_root=dataset_root,
                num_classes=num_classes,
                init=init,
                seed=seed,
                epochs=EPOCHS,
                warmup_epochs=10,
                batch_size=BATCH_SIZE,
                workers=WORKERS,
            )

!mkdir -p "$DRIVE_RESULTS_BACKUP"
!cp -r "$LOCAL_RESULTS"/* "$DRIVE_RESULTS_BACKUP"/
print("Backed up results to:", DRIVE_RESULTS_BACKUP)


/content/official_structured_initialization/pytorch-image-models-1.0.22


100%|██████████| 170M/170M [00:14<00:00, 12.1MB/s]


cifar10 train: saved 0/5000
cifar10 train: saved 2000/5000
cifar10 train: saved 4000/5000
cifar10 validation: saved 0/10000
cifar10 validation: saved 2000/10000
cifar10 validation: saved 4000/10000
cifar10 validation: saved 6000/10000
cifar10 validation: saved 8000/10000
Built cifar10 10% subset at: /content/official_structured_data/cifar10_10pct_seed1_imagefolder
Train images: 5000 | Validation images: 10000


100%|██████████| 169M/169M [00:13<00:00, 12.7MB/s]


cifar100 train: saved 0/5000
cifar100 train: saved 2000/5000
cifar100 train: saved 4000/5000
cifar100 validation: saved 0/10000
cifar100 validation: saved 2000/10000
cifar100 validation: saved 4000/10000
cifar100 validation: saved 6000/10000
cifar100 validation: saved 8000/10000
Built cifar100 10% subset at: /content/official_structured_data/cifar100_10pct_seed1_imagefolder
Train images: 5000 | Validation images: 10000

Starting experiment: cifar10_10pct_official_default_seed1_150ep_bs512
python -u train.py /content/official_structured_data/cifar10_10pct_seed1_imagefolder --train-split train --val-split validation --input-size 3 224 224 --mean 0.485 0.456 0.406 --std 0.229 0.224 0.225 --seed 1 --num-classes 10 --model vit_tiny_patch16_224 --model-kwargs img_size=224 weight_init=skip post_weight_init=default --model-ema-decay 0.9999 -j 8 -b 512 --lr 2e-3 --layer-decay 1.0 --warmup-lr 1e-6 --min-lr 1e-6 --weight-decay 0.05 --opt adamw --opt-eps 1e-8 --epochs 150 --sched cosine --warmup-e

In [ ]:
# Run 25% CIFAR-10 and 25% CIFAR-100 with official structured_initialization code.
# This creates ImageFolder-style datasets so we do not need train.py to support subset-frac.

%cd "$LOCAL_CODE"

import os
import shutil
import subprocess
import time
import random
from pathlib import Path

from torchvision.datasets import CIFAR10, CIFAR100


SUBSET_FRAC = 0.25
PCT_TAG = f"{int(SUBSET_FRAC * 100)}pct"

SEEDS = [1]          # change to [1, 2] if you want two seeds
EPOCHS = 150         # use 250 if you want same epoch budget as full-data runs
BATCH_SIZE = 512     # use 1024 if GPU memory allows
WORKERS = 8


def build_cifar_subset_imagefolder(dataset_name, seed=1, rebuild=False):
    dataset_name = dataset_name.lower()
    assert dataset_name in ["cifar10", "cifar100"]

    ds_class = CIFAR10 if dataset_name == "cifar10" else CIFAR100
    raw_root = Path(LOCAL_DATA) / "raw_torchvision"
    out_root = Path(LOCAL_DATA) / f"{dataset_name}_{PCT_TAG}_seed{seed}_imagefolder"

    if out_root.exists() and not rebuild:
        print(f"Using existing subset: {out_root}")
        return str(out_root)

    if out_root.exists():
        shutil.rmtree(out_root)

    train_set = ds_class(root=str(raw_root), train=True, download=True)
    test_set = ds_class(root=str(raw_root), train=False, download=True)
    classes = train_set.classes

    # Stratified subset per class.
    rng = random.Random(seed)
    by_class = {i: [] for i in range(len(classes))}
    for idx, label in enumerate(train_set.targets):
        by_class[label].append(idx)

    keep_indices = []
    for label, indices in by_class.items():
        rng.shuffle(indices)
        k = max(1, int(len(indices) * SUBSET_FRAC))
        keep_indices.extend(indices[:k])

    def save_split(dataset, indices, split_name):
        for n, idx in enumerate(indices):
            img, label = dataset[idx]
            class_dir = out_root / split_name / classes[label]
            class_dir.mkdir(parents=True, exist_ok=True)
            img.save(class_dir / f"{idx:06d}.png")

            if n % 2000 == 0:
                print(f"{dataset_name} {split_name}: saved {n}/{len(indices)}")

    save_split(train_set, keep_indices, "train")
    save_split(test_set, list(range(len(test_set))), "validation")

    print(f"Built {dataset_name} {PCT_TAG} subset at: {out_root}")
    print(f"Train images: {len(keep_indices)} | Validation images: {len(test_set)}")
    return str(out_root)


def run_official_imagefolder_subset(
    dataset_name,
    dataset_root,
    num_classes,
    init="default",
    seed=1,
    epochs=150,
    warmup_epochs=10,
    batch_size=512,
    workers=8,
):
    exp = f"{dataset_name}_{PCT_TAG}_official_{init}_seed{seed}_{epochs}ep_bs{batch_size}"

    cmd = [
        "python", "-u", "train.py", dataset_root,

        "--train-split", "train",
        "--val-split", "validation",

        "--input-size", "3", "224", "224",
        "--mean", "0.485", "0.456", "0.406",
        "--std", "0.229", "0.224", "0.225",

        "--seed", str(seed),
        "--num-classes", str(num_classes),
        "--model", "vit_tiny_patch16_224",
        "--model-kwargs",
        "img_size=224",
        "weight_init=skip",
        f"post_weight_init={init}",

        "--model-ema-decay", "0.9999",
        "-j", str(workers),
        "-b", str(batch_size),

        "--lr", "2e-3",
        "--layer-decay", "1.0",
        "--warmup-lr", "1e-6",
        "--min-lr", "1e-6",
        "--weight-decay", "0.05",
        "--opt", "adamw",
        "--opt-eps", "1e-8",
        "--epochs", str(epochs),
        "--sched", "cosine",
        "--warmup-epochs", str(warmup_epochs),
        "--amp",

        "--aa", "rand-m9-mstd0.5-inc1",
        "--cutmix", "1.0",
        "--mixup", "0.8",
        "--reprob", "0.25",
        "--smoothing", "0.1",
        "--drop", "0.0",
        "--color-jitter", "0.4",
        "--drop-path", "0.1",
        "--crop-pct", "0.875",
        "--pin-mem",

        "--checkpoint-hist", "1",
        "--recovery-interval", "0",

        "--output", LOCAL_RESULTS,
        "--experiment", exp,
    ]

    print("\n" + "=" * 90)
    print(f"Starting experiment: {exp}")
    print(" ".join(cmd))
    print("=" * 90 + "\n", flush=True)

    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    recent_lines = []
    last_progress_time = time.time()

    for line in process.stdout:
        line = line.rstrip()
        recent_lines.append(line)
        recent_lines = recent_lines[-8:]

        important = (
            "Train:" in line
            or "Test:" in line
            or "Epoch:" in line
            or "Acc@" in line
            or "loss" in line.lower()
            or "error" in line.lower()
            or "warning" in line.lower()
        )

        if important:
            print(line, flush=True)

        if time.time() - last_progress_time > 60:
            print("\n--- still running; recent output ---")
            for l in recent_lines:
                print(l)
            print("-----------------------------------\n", flush=True)
            last_progress_time = time.time()

    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"{exp} failed with exit code {return_code}")

    print(f"\nFinished experiment: {exp}\n", flush=True)


for seed in SEEDS:
    cifar10_root = build_cifar_subset_imagefolder("cifar10", seed=seed)
    cifar100_root = build_cifar_subset_imagefolder("cifar100", seed=seed)

    for dataset_name, dataset_root, num_classes in [
        ("cifar10", cifar10_root, 10),
        ("cifar100", cifar100_root, 100),
    ]:
        for init in ["default", "mimetic", "impulse"]:
            run_official_imagefolder_subset(
                dataset_name=dataset_name,
                dataset_root=dataset_root,
                num_classes=num_classes,
                init=init,
                seed=seed,
                epochs=EPOCHS,
                warmup_epochs=10,
                batch_size=BATCH_SIZE,
                workers=WORKERS,
            )

!mkdir -p "$DRIVE_RESULTS_BACKUP"
!cp -r "$LOCAL_RESULTS"/* "$DRIVE_RESULTS_BACKUP"/
print("Backed up results to:", DRIVE_RESULTS_BACKUP)


/content/official_structured_initialization/pytorch-image-models-1.0.22


100%|██████████| 170M/170M [00:16<00:00, 10.4MB/s]


cifar10 train: saved 0/12500
cifar10 train: saved 2000/12500
cifar10 train: saved 4000/12500
cifar10 train: saved 6000/12500
cifar10 train: saved 8000/12500
cifar10 train: saved 10000/12500
cifar10 train: saved 12000/12500
cifar10 validation: saved 0/10000
cifar10 validation: saved 2000/10000
cifar10 validation: saved 4000/10000
cifar10 validation: saved 6000/10000
cifar10 validation: saved 8000/10000
Built cifar10 25pct subset at: /content/official_structured_data/cifar10_25pct_seed1_imagefolder
Train images: 12500 | Validation images: 10000


100%|██████████| 169M/169M [00:13<00:00, 12.4MB/s]


cifar100 train: saved 0/12500
cifar100 train: saved 2000/12500
cifar100 train: saved 4000/12500
cifar100 train: saved 6000/12500
cifar100 train: saved 8000/12500
cifar100 train: saved 10000/12500
cifar100 train: saved 12000/12500
cifar100 validation: saved 0/10000
cifar100 validation: saved 2000/10000
cifar100 validation: saved 4000/10000
cifar100 validation: saved 6000/10000
cifar100 validation: saved 8000/10000
Built cifar100 25pct subset at: /content/official_structured_data/cifar100_25pct_seed1_imagefolder
Train images: 12500 | Validation images: 10000

Starting experiment: cifar10_25pct_official_default_seed1_150ep_bs512
python -u train.py /content/official_structured_data/cifar10_25pct_seed1_imagefolder --train-split train --val-split validation --input-size 3 224 224 --mean 0.485 0.456 0.406 --std 0.229 0.224 0.225 --seed 1 --num-classes 10 --model vit_tiny_patch16_224 --model-kwargs img_size=224 weight_init=skip post_weight_init=default --model-ema-decay 0.9999 -j 8 -b 512 --lr